In [1]:
import os
import gc
import torch
from pydub import AudioSegment
from faster_whisper import WhisperModel

def cleanup_all():
    gc.collect()

# Mac: no CUDA. Use CPU. If you later want to try MPS, use transformers for summarization.
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", torch.backends.mps.is_available())

# Choose a smaller model if you want faster/lower memory use: "tiny", "base", "small"
asr_model = WhisperModel("small", device="cpu", compute_type="int8")
print("ASR model loaded.")


/Users/sameerhussain/Desktop/fyp_asr/asr/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: False
MPS available: True
ASR model loaded.


In [4]:
audio_path = "/Users/sameerhussain/Desktop/fyp_asr/audio/doc_patient_convo.mp3"
audio = AudioSegment.from_file(audio_path)

# Convert to mono, 16kHz
audio = audio.set_channels(1).set_frame_rate(16000)
processed_path = "doc_patient_convo_mono16k.wav"
audio.export(processed_path, format="wav")
print("Audio converted to mono 16kHz.")


Audio converted to mono 16kHz.


In [5]:
os.makedirs("stream_transcripts", exist_ok=True)

segments, info = asr_model.transcribe(processed_path, beam_size=5)
transcript_text = " ".join([seg.text.strip() for seg in segments])

transcript_file = "stream_transcripts/full_transcript.txt"
with open(transcript_file, "w", encoding="utf-8") as f:
    f.write(transcript_text.strip())

cleanup_all()
print("Full transcript saved.")

/Users/sameerhussain/Desktop/fyp_asr/asr/lib/python3.10/site-packages/faster_whisper/feature_extractor.py:224: RuntimeWarning: divide by zero encountered in matmul
  mel_spec = self.mel_filters @ magnitudes
/Users/sameerhussain/Desktop/fyp_asr/asr/lib/python3.10/site-packages/faster_whisper/feature_extractor.py:224: RuntimeWarning: overflow encountered in matmul
  mel_spec = self.mel_filters @ magnitudes
/Users/sameerhussain/Desktop/fyp_asr/asr/lib/python3.10/site-packages/faster_whisper/feature_extractor.py:224: RuntimeWarning: invalid value encountered in matmul
  mel_spec = self.mel_filters @ magnitudes


Full transcript saved.


In [6]:
print("Unloading ASR model and freeing memory...")
del asr_model
cleanup_all()
print("ASR model unloaded, memory freed.")

Unloading ASR model and freeing memory...
ASR model unloaded, memory freed.


In [7]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

# Use MPS if available, else CPU
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

summarizer = pipeline("summarization", model=model, tokenizer=tokenizer, device=device)

summaries = []
for c in chunks:
    summaries.append(
        summarizer(c, max_length=250, min_length=80, do_sample=False)[0]["summary_text"]
    )

final_summary = " ".join(summaries)
print(final_summary)


Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 511/511 [00:00<00:00, 1742.49it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]   


KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"

In [9]:
import re

with open(transcript_file, "r", encoding="utf-8") as f:
    all_text = f.read()

# Split by sentence
sentences = re.split(r'(?<=[.!?]) +', all_text)

# Chunk ~200 words per chunk
chunks = []
current = ""
for s in sentences:
    if len(current.split()) + len(s.split()) > 200:
        chunks.append(current.strip())
        current = ""
    current += s + " "
if current:
    chunks.append(current.strip())

print(f"{len(chunks)} chunks created for summarization.")


3 chunks created for summarization.


In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

device = "mps" if torch.backends.mps.is_available() else "cpu"

model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# Required for BART in recent versions
model.generation_config.forced_bos_token_id = tokenizer.bos_token_id or 0

def summarize(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024).to(device)
    summary_ids = model.generate(
        **inputs,
        max_length=250,
        min_length=80,
        do_sample=False
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

summaries = [summarize(c) for c in chunks]
final_summary = " ".join(summaries)
print(final_summary)


Loading weights: 100%|██████████| 511/511 [00:00<00:00, 1759.22it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]   


Zahra: "I've had it for about three weeks now, and it just won't go away" "It's mostly dry, but sometimes, especially in the mornings, I cough up a little bit of phlegm" "I felt a bit run down, a little tired, and my chest feels tight, like a weight on my chest" "Do you have any history of asthma or allergies? No, none that I know of" Zara might have a touch of bronchitis. She should drink lots of fluids, especially warm liquids like tea with honey. Avoid irritants like smoke or strong perfumes. If you develop a high fever, over 102 degrees Fahrenheit, or if you have trouble breathing, please go to the emergency room immediately..com/Zara. For confidential support call the Samaritans on 08457 90 90 90, visit a local Samaritans branch or click here. Come back in three days if you have not improved. If you have improved, come back three days later. If not, go back to the previous page and try again next week. Back to Mail Online home. Back To The page you came from.back to the page you w